In [ ]:
import pandas as pd, requests, time
#pull complaints datab from cfpb api
# def fetch_cfpb_complaints():
#     base_url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
#     all_complaints = []
#     offset = 0
#     batch_size = 10  # Number of records to fetch per request
    
#     while True:
#         params = {
#             # 'company': 'USAA',  # Filter for USAA company
#             'size': batch_size,  # Maximum records per request (API limit)
#             'sort': 'created_date_desc',  # Sort by newest first
#             'format': 'json',
#             'from': offset
#         }
        
#         response = requests.get(base_url, params=params)
#         data = response.json()
#         print(data)

#         if not complaints:
#             break
        
#         all_complaints.extend(complaints)
#         offset += batch_size
#         time.sleep(1)  # To avoid hitting rate limits
    
#     return pd.DataFrame(all_complaints)
# df_complaints = fetch_cfpb_complaints()
# df_complaints
#Q: why is this not fetching any data?
#a: The 'company' filter is commented out, so it fetches all complaints, but the loop breaks immediately if no complaints are returned.
#q: HOW WOULD I FIX THE JSONDecodeError: Unterminated string starting at: line 1 column 1885187334 (char 1885187333)?
#a: This error typically occurs when the response from the API is not valid JSON. To fix this, you can add error handling to check if the response is valid before attempting to parse it. Here's an updated version of the code with error handling:
import pandas as pd, requests, time
#pull complaints datab from cfpb api
def fetch_cfpb_complaints(max_records=1):
    base_url = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
    all_complaints = []
    offset = 0
    batch_size = 1  # Number of records to fetch per request
    
    while True and len(all_complaints) <= max_records:
        params = {
            'company': 'UNITED SERVICES AUTOMOBILE ASSOCIATION',  # Filter for USAA company
            'size': batch_size,  # Maximum records per request (API limit)
            'sort': 'created_date_desc',  # Sort by newest first
            'format': 'json',
            'from': offset
        }
        
        response = requests.get(base_url, params=params)
        return_data = response.text
        return return_data 
    
#pull 1 record to test
data = fetch_cfpb_complaints()
data
#q: why is this running so slowly?
#a: The code is designed to fetch data in small batches (1 record at a time), which can be slow. Increasing the batch size will speed up the process.
#Q: shouldnt this just pull 1 record and be fast?
#a: Yes, pulling just 1 record should be fast. If it's still slow, it could be due to network latency or server response time from the API.4


In [13]:
import pandas as pd
dta = pd.read_csv('complaints.csv')
dta.shape


(11522175, 18)

In [15]:
#lowercase column names and replace spaces with underscores
dta.columns = dta.columns.str.lower().str.replace(' ', '_')
dta.head(2)

,date_received,product,sub-product,issue,sub-issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,consumer_consent_provided?,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response?,consumer_disputed?,complaint_id
0,2020-07-06,"Credit reporting, credit repair services, or o...",Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,Company has responded to the consumer and the ...,Experian Information Solutions Inc.,FL,346XX,NaN,Other,Web,2020-07-06,Closed with explanation,Yes,NaN,3730948
1,2025-10-14,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information is missing that should be on the r...,NaN,NaN,"EQUIFAX, INC.",TX,75062,NaN,NaN,Web,2025-10-14,In progress,Yes,NaN,16558024


In [ ]:
dta.head(2)
#convert date received to datetime
dta['date_received'] = pd.to_datetime(dta['date_received'])
#convert date_recieved to date only
dta['date_received'] = dta['date_received'].dt.date
dta['date_received'].head(2)


(9108625, 18)


,date_received,product,sub-product,issue,sub-issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,consumer_consent_provided?,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response?,consumer_disputed?,complaint_id
1,2025-10-14,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information is missing that should be on the r...,NaN,NaN,"EQUIFAX, INC.",TX,75062,NaN,NaN,Web,2025-10-14,In progress,Yes,NaN,16558024
2,2025-10-10,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",GA,30341,NaN,NaN,Web,2025-10-10,In progress,Yes,NaN,16507707


In [18]:
#keep only the lat 3 years of data
dta = dta[dta['date_received'] >= pd.to_datetime('2023-01-01').date()]
print(dta.shape)
dta.head(2)


(8308285, 18)


,date_received,product,sub-product,issue,sub-issue,consumer_complaint_narrative,company_public_response,company,state,zip_code,tags,consumer_consent_provided?,submitted_via,date_sent_to_company,company_response_to_consumer,timely_response?,consumer_disputed?,complaint_id
1,2025-10-14,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information is missing that should be on the r...,NaN,NaN,"EQUIFAX, INC.",TX,75062,NaN,NaN,Web,2025-10-14,In progress,Yes,NaN,16558024
2,2025-10-10,Credit reporting or other personal consumer re...,Credit reporting,Incorrect information on your report,Information belongs to someone else,NaN,NaN,"EQUIFAX, INC.",GA,30341,NaN,NaN,Web,2025-10-10,In progress,Yes,NaN,16507707


In [20]:
dta.company.value_counts().to_csv('complaint_companies_counts.csv')